# Question Answering

## Overview

Recall the overall workflow for retrieval augmented generation (RAG):

![overview.jpeg](attachment:overview.jpeg)

We discussed `Document Loading` and `Splitting` as well as `Storage` and `Retrieval`.

Let's load our vectorDB.

In [ ]:
# import os
# import openai
# import sys
# sys.path.append('../..')

# from dotenv import load_dotenv, find_dotenv
# _ = load_dotenv(find_dotenv()) # read local .env file

# openai.api_key  = os.environ['OPENAI_API_KEY']

The code below was added to assign the openai LLM version filmed until it is deprecated, currently in Sept 2023.
LLM responses can often vary, but the responses may be significantly different when using a different model version.

In [ ]:
# import datetime
# current_date = datetime.datetime.now().date()
# if current_date < datetime.date(2023, 9, 2):
#     llm_name = "gpt-3.5-turbo-0301"
# else:
#     llm_name = "gpt-3.5-turbo"
# print(llm_name)

In [ ]:
!pip install -q huggingface_hub
!pip install -qU langchain-community langchain_huggingface langchain_community langchain-huggingface langchainhub langchain_chroma
!pip install -q transformers einops accelerate bitsandbytes
! pip install -q pypdf PyPDF2 tiktoken

In [ ]:
from langchain.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.document_loaders import PyPDFLoader
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          pipeline, BitsAndBytesConfig)
import PyPDF2
import os
os.environ["HF_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"

In [ ]:
def merge_pdfs(input_paths, output_path):
    merger = PyPDF2.PdfMerger()
    for path in input_paths:
        with open(path, 'rb') as file:
            merger.append(file)
    with open(output_path, 'wb') as output_file:
        merger.write(output_file)

In [ ]:
pdf_files = ["/content/machinelearning-lecture01.pdf",
             "/content/MachineLearning-Lecture02.pdf",
             "/content/MachineLearning-Lecture03.pdf"]

output_filename = "/content/merged.pdf"

merge_pdfs(pdf_files, output_filename)

print(f"Merged PDF saved to: {output_filename}")

Merged PDF saved to: /content/merged.pdf


In [ ]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config)
tokenizer = AutoTokenizer.from_pretrained(model_id)

pipe = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.1,
    max_new_tokens=500,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
)
llm = HuggingFacePipeline(pipeline=pipe)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [ ]:
loader = PyPDFLoader("/content/merged.pdf")
docs = loader.load()
chunk_size = 1500
chunk_overlap = 150
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                               chunk_overlap=chunk_overlap,
                                               add_start_index = True)
splits = text_splitter.split_documents(docs)

In [ ]:
model_name = "BAAI/bge-m3"
embedding = HuggingFaceEmbeddings(model_name=model_name)
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
print(vectordb._collection.count())

151


In [ ]:
question = "What are major topics for this class?"
docs = vectordb.similarity_search(question,k=3)
len(docs)

3

### RetrievalQA chain

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever()
)

In [ ]:
result = qa_chain.invoke({"query": question})
result["result"]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


' Based on the provided text, the major topics for this class seem to include:\n\n1. Linear Regression\n2. Gradient Descent\n3. Normal Equations\n4. Probability and Statistics (reviewing basic concepts)\n5. Linear Algebra (reviewing basic concepts)\n\nNote that the instructor mentions that the class will cover these topics, but also mentions that there will be additional material covered in the discussion sections, which may include reviewing prerequisites, going over extensions, and providing a refresher course on certain topics.'

### Prompt

In [ ]:
from langchain.prompts import PromptTemplate

# Build prompt
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum. Keep the answer as concise as possible.
Always say "thanks for asking!" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

In [ ]:
# Run chain
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

In [ ]:
question = "Is probability a class topic?"
result = qa_chain.invoke({"query": question})
result["result"]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


' Yes, according to the text, probability is a class topic. The instructor mentions that they will review probability and statistics in the discussion sections for those who need a refresher.\n\nThanks for asking!'

In [ ]:
result["source_documents"][0]

Document(metadata={'page': 48, 'source': '/content/merged.pdf', 'start_index': 2839}, page_content='Gaussian densities that I wrote down just now, right? Okay?  \nThen in parts of notation, I guess, I define this term here to be the likelihood of theta. \nAnd the likely of theta is just the probability of the data Y, right? Given X and prioritized \nby theta. To test the likelihood and probability are often confused. So the likelihood of \ntheta is the same thing as the probability of the data you saw. So likely and probably are, \nsort of, the same thing. Except that when I use the term likelihood I’m trying to \nemphasize that I’m taking this thing and viewing it as a function of theta. Okay? So \nlikelihood and for probability, they’re really the same thing except that when I want to \nview this thing as a function of theta holding X and Y fix are then called likelihood. \nOkay? So hopefully you hear me say the likelihood of the parameters and the probability')

### RetrievalQA chain types

In [ ]:
qa_chain_mr = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(),
    chain_type="map_reduce"
)

In [ ]:
result = qa_chain_mr({"query": question})

<ipython-input-17-ed06c68e021f>:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain_mr({"query": question})
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (2231 > 1024). Running this sequence through the model will result in indexing errors
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
result["result"]

' Yes, probability is a class topic.'

If you wish to experiment on the `LangSmith platform` (previously known as LangChain Plus):

 * Go to [LangSmith](https://www.langchain.com/langsmith) and sign up
 * Create an API key from your account's settings
 * Use this API key in the code below   
 * uncomment the code  
 Note, the endpoint in the video differs from the one below. Use the one below.

In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.langchain.plus"
os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGSMITH_API_KEY" # replace dots with your api key

In [ ]:
qa_chain_mr = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(),
    chain_type="map_reduce"
)
result = qa_chain_mr({"query": question})
result["result"]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


' Yes, probability is a class topic.'

In [ ]:
qa_chain_mr = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(),
    chain_type="refine"
)
result = qa_chain_mr({"query": question})
result["result"]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


''

### Limitações do RetrievalQA

O QA falha em preservar o histórico de conversação.

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever()
)

question = "Is probability a class topic?"
result = qa_chain({"query": question})
result["result"]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


' Yes, according to the text, probability is a class topic. The instructor mentions that they will review probability and statistics in the discussion sections for students who need a refresher. Additionally, the instructor uses probability-related concepts such as likelihood and probability density functions throughout the lecture.'

In [ ]:
question = "why are those prerequesites needed?"
result = qa_chain({"query": question})
result["result"]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


' The instructor assumes that students have prior knowledge in basic probability and statistics, including concepts such as random variables, expectations, variances, and matrix operations. This is because machine learning builds upon these mathematical foundations. Additionally, the instructor assumes familiarity with basic linear algebra, including matrix multiplication, inverses, and eigenvectors, as these concepts are crucial for understanding many machine learning algorithms. Finally, the instructor assumes students have basic computer science knowledge, including understanding of big O notation and data structures, as they will be working with programming languages like MATLAB or Octave. These prerequisites are necessary to ensure that students have a solid foundation in mathematics and computer science before diving into the world of machine learning.'

Nota: A resposta do LLM varia. Algumas respostas **incluem** uma referência à probabilidade que pode ser obtida de documentos referenciados. O ponto é simplesmente que o modelo não tem acesso a perguntas ou respostas anteriores, isso será abordado na próxima seção.